## 1. **Import Libraries**

In [ ]:
import os
import json
import random
import logging
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import albumentations as A

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger
)

# Suppress TF noise
logging.getLogger('tensorflow').setLevel(logging.ERROR)
tf.get_logger().setLevel('ERROR')

print("TensorFlow version :", tf.__version__)
print("GPU Available       :", len(tf.config.list_physical_devices('GPU')) > 0)
print("Albumentations ver  :", A.__version__)


## 2. **Data Loading and Preprocessing**

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
BASE_DIR   = Path('/content/drive/MyDrive/Brain-Tumor-Mri-Deep-Learning')
JSON_PATH  = BASE_DIR / 'DATA.json'
IMG_DIR    = BASE_DIR / 'data' / 'raw'       # folder chứa ảnh gốc

PROJECT_ROOT = Path('.')
MODEL_DIR  = PROJECT_ROOT / 'models'
REPORT_DIR = PROJECT_ROOT / 'reports'
FIGURE_DIR = REPORT_DIR / 'figures'
LOG_DIR    = PROJECT_ROOT / 'logs'

for d in [MODEL_DIR, REPORT_DIR, FIGURE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Hyper-parameters ──────────────────────────────────────────────────────
IMG_SIZE       = 256          # pixels (ảnh MRI 512→resize 256 cho nhanh)
BATCH_SIZE     = 16
EPOCHS_PHASE1  = 40           # frozen backbone, train head
EPOCHS_PHASE2  = 15           # full fine-tune
HEATMAP_SIGMA  = 15           # Gaussian sigma for lesion heatmap
SEED           = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"JSON path   : {JSON_PATH}")
print(f"Images dir  : {IMG_DIR}")


In [ ]:
# ── Load metadata from JSON ───────────────────────────────────────────────
def load_dataset_info(json_path, img_dir):
    """Parse DATA.json, resolve image paths, extract class & lesion point."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    dataset_info = []
    classes_set  = set()
    missing      = 0

    for rel_path, info in data.items():
        safe_path    = rel_path.replace('\\', os.sep).replace('/', os.sep)
        full_img_path = Path(img_dir) / safe_path

        if not full_img_path.exists():
            missing += 1
            continue

        class_name = info['class']
        classes_set.add(class_name)

        # Normal classes have no lesion → point set to None
        if 'Normal' in class_name:
            point = None
        else:
            point = (info['point']['x'], info['point']['y'])

        dataset_info.append({
            'img_path': str(full_img_path),
            'class'   : class_name,
            'point'   : point
        })

    classes_list  = sorted(list(classes_set))
    class_to_idx  = {cls: idx for idx, cls in enumerate(classes_list)}

    print(f"Total images loaded : {len(dataset_info)}")
    print(f"Missing / skipped   : {missing}")
    print(f"Classes identified  : {len(classes_list)}")
    return dataset_info, class_to_idx, classes_list

dataset_info, class_to_idx, CLASS_NAMES = load_dataset_info(JSON_PATH, IMG_DIR)
NUM_CLASSES = len(CLASS_NAMES)


In [ ]:
# ── Class distribution ────────────────────────────────────────────────────
from collections import Counter

count_map = Counter(d['class'] for d in dataset_info)
dist_df   = pd.DataFrame(count_map.items(), columns=['class_name', 'image_count'])
dist_df   = dist_df.sort_values('class_name').reset_index(drop=True)

display(dist_df)
print("Total images:", dist_df['image_count'].sum())

plt.figure(figsize=(16, 5))
plt.bar(dist_df['class_name'], dist_df['image_count'])
plt.title('Full Dataset – Class Distribution')
plt.xlabel('Class'); plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'class_distribution.png', dpi=150)
plt.show()


In [ ]:
# ── Train / Validation split (stratified 80/20) ───────────────────────────
train_info, val_info = train_test_split(
    dataset_info,
    test_size=0.2,
    random_state=SEED,
    stratify=[d['class'] for d in dataset_info]
)

print(f"Train samples : {len(train_info)}")
print(f"Val   samples : {len(val_info)}")


In [ ]:
# ── Gaussian heatmap generator ────────────────────────────────────────────
def generate_gaussian_heatmap(size, center, sigma):
    """Produce a 2D Gaussian centred on the lesion marker."""
    heatmap = np.zeros((size, size), dtype=np.float32)
    if center is None:
        return heatmap                     # Normal class → empty map

    x0, y0 = int(center[0]), int(center[1])
    if not (0 <= x0 < size and 0 <= y0 < size):
        return heatmap                     # out-of-bounds guard

    x = np.arange(0, size, dtype=float)
    y = np.arange(0, size, dtype=float)[:, np.newaxis]
    heatmap = np.exp(-4 * np.log(2) * ((x - x0)**2 + (y - y0)**2) / sigma**2)
    return heatmap.astype(np.float32)


# ── Custom data generator ─────────────────────────────────────────────────
class BrainTumorDataGenerator(Sequence):
    """
    Yields (images, {class_output, heatmap_output}) batches.
    Augmentation is handled by Albumentations so keypoints are
    transformed consistently with the image.
    """

    def __init__(self, dataset_info, class_to_idx, batch_size,
                 img_size, augment=False, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.dataset_info = dataset_info
        self.class_to_idx = class_to_idx
        self.num_classes  = len(class_to_idx)
        self.batch_size   = batch_size
        self.img_size     = img_size
        self.shuffle      = shuffle

        kp_params = A.KeypointParams(format='xy', remove_invisible=False)

        if augment:
            self.transform = A.Compose([
                A.Resize(img_size, img_size),
                A.HorizontalFlip(p=0.5),
                A.RandomBrightnessContrast(
                    brightness_limit=0.15, contrast_limit=0.15, p=0.4),
                A.Affine(
                    translate_percent={'x': (-0.05, 0.05), 'y': (-0.05, 0.05)},
                    scale={'x': (0.95, 1.05), 'y': (0.95, 1.05)},
                    rotate=0, p=0.3),
                A.ElasticTransform(alpha=1, sigma=50, p=0.2)
            ], keypoint_params=kp_params)
        else:
            self.transform = A.Compose(
                [A.Resize(img_size, img_size)],
                keypoint_params=kp_params)

        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.dataset_info) / self.batch_size))

    def __getitem__(self, index):
        idxs      = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        batch_info = [self.dataset_info[k] for k in idxs]

        X         = np.empty((self.batch_size, self.img_size, self.img_size, 3), np.float32)
        y_class   = np.empty((self.batch_size, self.num_classes), np.float32)
        y_heatmap = np.empty((self.batch_size, self.img_size, self.img_size, 1), np.float32)

        for i, info in enumerate(batch_info):
            img = cv2.imread(info['img_path'])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            keypoints = [info['point']] if info['point'] is not None else []
            result    = self.transform(image=img, keypoints=keypoints)

            X[i] = result['image'] / 255.0

            cls_idx     = self.class_to_idx[info['class']]
            y_class[i]  = tf.keras.utils.to_categorical(cls_idx, self.num_classes)

            new_pt      = result['keypoints'][0] if result['keypoints'] else None
            hm          = generate_gaussian_heatmap(self.img_size, new_pt, HEATMAP_SIGMA)
            y_heatmap[i] = np.expand_dims(hm, axis=-1)

        return X, {'class_output': y_class, 'heatmap_output': y_heatmap}

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.dataset_info))
        if self.shuffle:
            np.random.shuffle(self.indexes)


train_gen = BrainTumorDataGenerator(
    train_info, class_to_idx, BATCH_SIZE, IMG_SIZE, augment=True, shuffle=True)
val_gen   = BrainTumorDataGenerator(
    val_info,   class_to_idx, BATCH_SIZE, IMG_SIZE, augment=False, shuffle=False)

print(f"Train batches : {len(train_gen)}")
print(f"Val   batches : {len(val_gen)}")


In [ ]:
# ── Visualise sample batch ────────────────────────────────────────────────
X_sample, Y_sample = train_gen[0]
labels_sample = Y_sample['class_output']

plt.figure(figsize=(12, 12))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_sample[i])
    cls_idx  = np.argmax(labels_sample[i])
    plt.title(CLASS_NAMES[cls_idx], fontsize=7)
    plt.axis('off')

plt.suptitle('Sample Augmented Training Images', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'preprocessed_sample_images.png', dpi=150)
plt.show()

print('Image batch shape   :', X_sample.shape)
print('Label batch shape   :', labels_sample.shape)
print('Heatmap batch shape :', Y_sample['heatmap_output'].shape)
print('Pixel range         : [{:.2f}, {:.2f}]'.format(X_sample.min(), X_sample.max()))


## **Build Model – EfficientNetV2S Dual-Head (Classification + Heatmap)**

### Architecture rationale

| Component | Choice | Why |
|-----------|--------|-----|
| Backbone | EfficientNetV2S (ImageNet) | SOTA accuracy/param; fused-MBConv blocks train fast on P100 |
| Classification head | GAP → Dropout(0.4) → Dense(30, softmax) | Lightweight head avoids overfitting on ~9k train images |
| Heatmap head | Conv2DTranspose ×5 → sigmoid | Leverages lesion-point metadata; acts as auxiliary regulariser |
| Loss | CE (weight 1.0) + MSE heatmap (weight 0.5) | Dual supervision improves feature localisation |
| Metrics | accuracy, precision, recall, Top-3 accuracy | Clinical relevance: top-3 covers differential diagnosis |
| Phase 1 LR | 1e-4 Adam + ReduceLROnPlateau | Stable head training with frozen backbone |
| Phase 2 LR | 1e-5 Adam (all layers) | Gentle fine-tune avoids catastrophic forgetting |
| Regularisation | Dropout 0.4, EarlyStopping, class weights | Multi-level defence against overfitting |

### Two-phase training
1. **Phase 1 (40 epochs max)**: Only last 50 backbone layers trainable → train head + heatmap decoder.
2. **Phase 2 (15 epochs max)**: All layers unfrozen with 10× lower LR → deep fine-tune.


In [ ]:
# ── EpochSummaryReport callback ───────────────────────────────────────────
class EpochSummaryReport(tf.keras.callbacks.Callback):
    """Pretty epoch summary matching the reference training log style."""
    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        acc     = logs.get('class_output_accuracy', 0) * 100
        prec    = logs.get('class_output_precision', 0) * 100
        rec     = logs.get('class_output_recall', 0) * 100
        loss    = logs.get('class_output_loss', 0)
        val_acc = logs.get('val_class_output_accuracy', 0) * 100
        val_pre = logs.get('val_class_output_precision', 0) * 100
        val_rec = logs.get('val_class_output_recall', 0) * 100
        val_los = logs.get('val_class_output_loss', 0)

        print(f"\n{'='*60}")
        print(f"🏁 EPOCH {epoch + 1} SUMMARY 🏁")
        print(f"{'-'*60}")
        print(f"📊 TRAINING PERFORMANCE:")
        print(f"   • Accuracy: {acc:.2f}%, Precision: {prec:.2f}%, "
              f"Sensitivity/Recall: {rec:.2f}%, Loss: {loss:.4f}")
        print(f"🩺 VALIDATION PERFORMANCE:")
        print(f"   • Accuracy: {val_acc:.2f}%, Precision: {val_pre:.2f}%, "
              f"Sensitivity/Recall: {val_rec:.2f}%, Loss: {val_los:.4f}")
        print(f"{'='*60}\n")


In [ ]:
# ── Dual-head model ───────────────────────────────────────────────────────
def build_dual_head_model(input_size, num_classes):
    """
    EfficientNetV2S backbone with:
      - Classification head: GAP → Dropout → Dense(softmax)
      - Heatmap head       : Conv2DTranspose × 5 → sigmoid (same spatial size as input)
    """
    inputs     = tf.keras.Input(shape=(input_size, input_size, 3), name='input_image')
    base_model = EfficientNetV2S(
        include_top=False, weights='imagenet', input_tensor=inputs)

    # Freeze all except top 50 layers for Phase 1
    for layer in base_model.layers[:-50]:
        layer.trainable = False

    x = base_model.output

    # ── Classification head ───────────────────────────────────────────────
    gap         = layers.GlobalAveragePooling2D(name='gap')(x)
    gap         = layers.Dropout(0.4, name='drop_cls')(gap)
    class_out   = layers.Dense(
        num_classes, activation='softmax', name='class_output')(gap)

    # ── Heatmap decoder (5× transposed conv to match input resolution) ────
    h = layers.Conv2DTranspose(256, (3,3), strides=2, padding='same',
                               activation='relu', name='up1')(x)
    h = layers.Conv2DTranspose(128, (3,3), strides=2, padding='same',
                               activation='relu', name='up2')(h)
    h = layers.Conv2DTranspose(64,  (3,3), strides=2, padding='same',
                               activation='relu', name='up3')(h)
    h = layers.Conv2DTranspose(32,  (3,3), strides=2, padding='same',
                               activation='relu', name='up4')(h)
    heatmap_out = layers.Conv2DTranspose(1, (3,3), strides=2, padding='same',
                               activation='sigmoid', name='heatmap_output')(h)

    model = models.Model(inputs=inputs, outputs=[class_out, heatmap_out],
                         name='BrainTumor_DualHead_EfficientNetV2S')
    return model, base_model


model, base_model = build_dual_head_model(IMG_SIZE, NUM_CLASSES)
model.summary()

trainable   = sum(tf.size(v).numpy() for v in model.trainable_variables)
total       = model.count_params()
print(f"\nTotal params     : {total:,}")
print(f"Trainable params : {trainable:,}")


## **Phase 1 – Head Training (last 50 backbone layers + heads)**

In [ ]:
# ── Compile ───────────────────────────────────────────────────────────────
METRICS_CLS = [
    'accuracy',
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')
]

def compile_model(model, lr):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss={
            'class_output'  : 'categorical_crossentropy',
            'heatmap_output': 'mean_squared_error'
        },
        loss_weights={'class_output': 1.0, 'heatmap_output': 0.5},
        metrics={'class_output': METRICS_CLS, 'heatmap_output': []}
    )

compile_model(model, lr=1e-4)

# ── Paths ─────────────────────────────────────────────────────────────────
best_p1_path   = MODEL_DIR / 'brain_tumor_phase1_best.keras'
best_p2_path   = MODEL_DIR / 'brain_tumor_finetuned_best.keras'
log_p1_path    = LOG_DIR   / 'phase1_training_log.csv'
log_p2_path    = LOG_DIR   / 'phase2_training_log.csv'

callbacks_p1 = [
    ModelCheckpoint(
        filepath=best_p1_path,
        monitor='val_class_output_accuracy',
        save_best_only=True, mode='max', verbose=1),
    EarlyStopping(
        monitor='val_class_output_accuracy',
        patience=10, restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(
        monitor='val_class_output_loss',
        factor=0.5, patience=4, min_lr=1e-7, verbose=1),
    CSVLogger(filename=log_p1_path, append=False),
    EpochSummaryReport()
]

print('=== PHASE 1: Head training (backbone mostly frozen) ===')
history_p1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks_p1
)


## **Phase 2 – Full Fine-tuning (all layers)**

In [ ]:
# Load best Phase-1 checkpoint, then unfreeze everything
model.load_weights(best_p1_path)

for layer in model.layers:
    layer.trainable = True

compile_model(model, lr=1e-5)    # 10× lower LR for safe fine-tune

callbacks_p2 = [
    ModelCheckpoint(
        filepath=best_p2_path,
        monitor='val_class_output_accuracy',
        save_best_only=True, mode='max', verbose=1),
    EarlyStopping(
        monitor='val_class_output_accuracy',
        patience=6, restore_best_weights=True, mode='max', verbose=1),
    CSVLogger(filename=log_p2_path, append=False),
    EpochSummaryReport()
]

print('=== PHASE 2: Full fine-tuning (all layers unfrozen) ===')
history_p2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks_p2
)

print("Training complete. Best fine-tuned model saved to:", best_p2_path)


## **Plot Training History**

In [ ]:
def plot_training_history(history, title_prefix, save_path):
    sns.set_theme(style='whitegrid')
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # Accuracy
    axes[0].plot(history.history['class_output_accuracy'],
                 label='Train', color='steelblue', linewidth=2)
    axes[0].plot(history.history['val_class_output_accuracy'],
                 label='Validation', color='darkorange', linewidth=2)
    axes[0].set_title(f'{title_prefix} – Accuracy', fontsize=14)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].grid(True)

    # Loss
    axes[1].plot(history.history['class_output_loss'],
                 label='Train', color='crimson', linestyle='--', linewidth=2)
    axes[1].plot(history.history['val_class_output_loss'],
                 label='Validation', color='seagreen', linewidth=2)
    axes[1].set_title(f'{title_prefix} – Loss', fontsize=14)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(True)

    # Recall
    axes[2].plot(history.history['class_output_recall'],
                 label='Train', color='mediumpurple', linewidth=2)
    axes[2].plot(history.history['val_class_output_recall'],
                 label='Validation', color='magenta', linewidth=2)
    axes[2].set_title(f'{title_prefix} – Recall', fontsize=14)
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Recall')
    axes[2].legend(); axes[2].grid(True)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()

plot_training_history(history_p1, 'Phase 1 (Head Only)',
                      FIGURE_DIR / 'graphs_phase1.png')
plot_training_history(history_p2, 'Phase 2 (Fine-tune)',
                      FIGURE_DIR / 'graphs_phase2_finetuning.png')


## **Evaluation & Metrics**

In [ ]:
# Load best fine-tuned model
best_model = tf.keras.models.load_model(best_p2_path)
print("Loaded best model from:", best_p2_path)


In [ ]:
# ── Predict on validation set ─────────────────────────────────────────────
preds = best_model.predict(val_gen, verbose=1)   # returns [class_probs, heatmaps]
y_pred_prob = preds[0]
y_pred      = np.argmax(y_pred_prob, axis=1)

# True labels from dataset_info (respects generator ordering)
n_samples   = len(val_gen) * BATCH_SIZE
y_true      = [class_to_idx[val_info[i]['class']] for i in range(n_samples)]

print('y_true shape:', np.array(y_true).shape)
print('y_pred shape:', y_pred.shape)


In [ ]:
# ── Scalar metrics ────────────────────────────────────────────────────────
accuracy           = accuracy_score(y_true, y_pred)
precision_macro    = precision_score(y_true, y_pred, average='macro',    zero_division=0)
recall_macro       = recall_score(y_true,    y_pred, average='macro',    zero_division=0)
f1_macro           = f1_score(y_true,        y_pred, average='macro',    zero_division=0)
precision_weighted = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall_weighted    = recall_score(y_true,    y_pred, average='weighted', zero_division=0)
f1_weighted        = f1_score(y_true,        y_pred, average='weighted', zero_division=0)

metrics_summary = {
    'model_name'        : 'EfficientNetV2S DualHead (fine-tuned)',
    'image_size'        : IMG_SIZE,
    'batch_size'        : BATCH_SIZE,
    'epochs_phase1'     : len(history_p1.history['class_output_loss']),
    'epochs_phase2'     : len(history_p2.history['class_output_loss']),
    'val_accuracy'      : float(accuracy),
    'precision_macro'   : float(precision_macro),
    'recall_macro'      : float(recall_macro),
    'f1_macro'          : float(f1_macro),
    'precision_weighted': float(precision_weighted),
    'recall_weighted'   : float(recall_weighted),
    'f1_weighted'       : float(f1_weighted)
}

metrics_df = pd.DataFrame([metrics_summary])
display(metrics_df)
metrics_df.to_csv(REPORT_DIR / 'efficientnet_dualhead_metrics.csv', index=False)


In [ ]:
# ── Classification report ─────────────────────────────────────────────────
print('\n===== Classification Report =====')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))


In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(24, 10))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)', fontsize=14)
axes[0].set_xlabel('Predicted Class'); axes[0].set_ylabel('Real Class')
axes[0].tick_params(axis='x', rotation=90); axes[0].tick_params(axis='y', rotation=0)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Confusion Matrix (normalised)', fontsize=14)
axes[1].set_xlabel('Predicted Class'); axes[1].set_ylabel('Real Class')
axes[1].tick_params(axis='x', rotation=90); axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'efficientnet_confusion_matrix.png', dpi=150)
plt.show()


In [ ]:
# ── Visualise heatmap predictions ────────────────────────────────────────
heatmaps   = preds[1]   # shape: (N, IMG_SIZE, IMG_SIZE, 1)
X_vis, _   = val_gen[0]

fig, axes = plt.subplots(3, 6, figsize=(20, 10))

for i in range(6):
    img = X_vis[i]
    hm  = heatmaps[i, :, :, 0]

    axes[0, i].imshow(img)
    axes[0, i].set_title(CLASS_NAMES[y_pred[i]], fontsize=7)
    axes[0, i].axis('off')

    axes[1, i].imshow(hm, cmap='hot', vmin=0, vmax=1)
    axes[1, i].set_title('Predicted Heatmap', fontsize=7)
    axes[1, i].axis('off')

    axes[2, i].imshow(img)
    axes[2, i].imshow(hm, cmap='jet', alpha=0.45, vmin=0, vmax=1)
    axes[2, i].set_title('Overlay', fontsize=7)
    axes[2, i].axis('off')

plt.suptitle('Lesion Localisation Heatmaps', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'heatmap_predictions.png', dpi=150)
plt.show()
